# Debug a Trial Experiment

In [1]:
from run_experiment import *
import experiment_config

xFormers not available
/unav/venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
xFormers not available


Seed set to: 42 (type: <class 'int'>)


/home/nattachart.tak/PhD/Trial_New_UNav/UNav/unav/core/third_party/LightGlue/lightglue/lightglue.py:15: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


In [2]:
# import argparse
# parser = argparse.ArgumentParser()
# parser.add_argument('-a', '--algorithm', type=str, default=, )
# parser.add_argument('-x', '--experiment', type=str, default=, help='Experiment config filepath for experimental VPR localization.')
# args = parser.parse_args()
class A:
    experiment = "/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/Config__v2-1_c/config.json"
    algorithm = "CricaVPR"
args = A()
config_exp = experiment_config.import_config(fn=args.experiment)
config_exp = update_dict(args.algorithm, config_exp)

DATA_FINAL_ROOT = config_exp.get('data_final_root', "/mnt/data/UNav-IO/data")
FEATURE_MODEL = args.algorithm #onfig_exp.get('global_descriptor_model', "DinoV2Salad")
config_exp['global_descriptor_model'] = args.algorithm
LOCAL_FEATURE_MODEL = config_exp.get('local_feature_model', "superpoint+lightglue")
PLACES = config_exp.get('places')
# , {
#                 "New_York_City": {
#                     "LightHouse": ["3_floor", "4_floor", "6_floor"]
#                 }
#             }

config = UNavConfig(
    data_final_root=DATA_FINAL_ROOT,
    places=PLACES,
    global_descriptor_model=FEATURE_MODEL,
    local_feature_model=LOCAL_FEATURE_MODEL
)
localizor_config = config.localizer_config
localizer = UNavLocalizer(localizor_config)
localizer.load_maps_and_features()


r = []
groundtruth_df = pd.read_csv(config_exp['ground_truth_img_list'])
image_filepaths = groundtruth_df[config_exp['img_list_attr']].apply(lambda x : f"{config_exp['path_to_images']}/{x}.{config_exp['img_ext']}").tolist()
config_exp['num_trials'] = 1000
#image_filepaths = ["/mnt/data/UNav-IO/test/photos/LightHouse/3-1.jpg"]
for trial_num in range(config_exp['num_trials']):
    for image_filepath in image_filepaths:
        if image_filepath not in ["/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000087_pitch00_yaw17.png",
                                 "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000268_pitch00_yaw02.png",
                                 "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000051_pitch00_yaw15.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000146_pitch00_yaw07.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000281_pitch00_yaw12.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000143_pitch00_yaw07.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000023_pitch00_yaw17.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000043_pitch00_yaw08.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000224_pitch00_yaw02.png"
                                ]:
        # if image_filepath != "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000268_pitch00_yaw02.png":
            continue
        img = get_image(image_filepath)
        cx,cy,ang = None,None,None
        error = None
        time_start = ExperimentTime.get_time_ms()
        # try:
        global_feat, local_feat_dict = localizer.extract_query_features(img)
        top_candidates = localizer.vpr_retrieve(global_feat, top_k=50)
        candidates_data = localizer.get_candidates_data(top_candidates)
        #print(candidates_data)
        best_map_key, pnp_pairs = get_best_map_key(localizer, local_feat_dict, candidates_data)
        refinement_queue = {best_map_key: {"pairs": [], "initial_poses": [], "pps": []}}
        time_start_multiframe = ExperimentTime.get_time_ms()
        refine_result = localizer.multi_frame_pose_refine(
            pnp_pairs, img.shape, refinement_queue[best_map_key]
        )
        print(f'Multiframe time (ms): {ExperimentTime.get_time_ms_duration(time_start_multiframe)}')

        cx,cy,ang = get_pose(localizer, best_map_key, refine_result)
        # except Exception as e:
        #     error = str(e)
        total_time = ExperimentTime.get_time_ms_duration(time_start)
        r += [
            {'coordx':cx,
                'coordy':cy,
                'angle':ang,
                'time_ms':total_time,
                'image_fn':image_filepath,
                'trial_num':trial_num,
                'error':error,
            }]

root_dir = config_exp['root_dir']
filename_parts = config_exp['results_fn'].split('.')
results_fn = f'{".".join(filename_parts[:-1])}_{args.algorithm}.{filename_parts[-1]}'

out_path = config_exp['results_out_path']
result_file = f'{root_dir}/{out_path}/{results_fn}'
pd.DataFrame.from_records(r).to_excel(result_file)
print(f"==== Predicted coordinates were saved to '{result_file}'. ====")

print("==== Calculate distance errors ====")
distance_error_fn = distance_error.save_calculated_distance_error(config_exp, results_fn)
print(f"==== Distance errors were saved to {distance_error_fn} ====")

[INFO] Initializing models: Local -> superpoint+lightglue | Global -> CricaVPR


/home/nattachart.tak/PhD/Trial_New_UNav/UNav/unav/core/third_party/SuperPoint_SuperGlue/extractors/SuperGluePretrainedNetwork/models/superpoint.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this

Loaded SuperPoint model


/unav/venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/home/nattachart.tak/PhD/Trial_New_UNav/UNav/unav/core/third_party/CricaVPR/util.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.

CricaVPR loaded from /home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/parameters/CricaVPR/ckpts/CricaVPR.pth successfully!
[✓] Loaded COLMAP model for ('Mahidol_University', 'ICT', '1.1_CricaVPR'): 6030 frames
[✓] Loaded global features for ('Mahidol_University', 'ICT', '1.1_CricaVPR'): 6030 images
[✓] Loaded transform matrix for ('Mahidol_University', 'ICT', '1.1_CricaVPR'): shape=(2, 4)
[INFO] All map and feature loading complete.


/unav/venv/lib/python3.10/site-packages/torch/nested/__init__.py:107: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return torch._nested_tensor_from_tensor_list(ts, dtype, None, device, None)


Best map key: ('Mahidol_University', 'ICT', '1.1_CricaVPR')
Number of candidates after local matching: 34
PnP pairs 2D shape: (2873, 2) 3D shape: (2873, 3)
Multiframe time (ms): 173
Floorplan Pose (x, y, angle): {'xy': array([1773.35234321, 1334.18245367]), 'ang': 206.84852329907022}
Best map key: ('Mahidol_University', 'ICT', '1.1_CricaVPR')
Number of candidates after local matching: 15
PnP pairs 2D shape: (980, 2) 3D shape: (980, 3)
Multiframe time (ms): 75
Floorplan Pose (x, y, angle): {'xy': array([ 790.6436558 , 1197.33295553]), 'ang': 155.1618149169763}
Best map key: ('Mahidol_University', 'ICT', '1.1_CricaVPR')
Number of candidates after local matching: 47
PnP pairs 2D shape: (4233, 2) 3D shape: (4233, 3)
Multiframe time (ms): 368
Floorplan Pose (x, y, angle): {'xy': array([1491.08625536, 1343.58427846]), 'ang': 110.64666472428515}
Best map key: ('Mahidol_University', 'ICT', '1.1_CricaVPR')
Number of candidates after local matching: 50
PnP pairs 2D shape: (10280, 2) 3D shape: (1

ValueError: Ransac failed to converge on a (stochastic) consensus after 3 attempts with error 'NoneType' object is not subscriptable